# Trading Research Engine — real data, start to finish

This notebook runs the whole system on **real market prices**. Colab has open
network access, so the exchange and market-data APIs that a Claude Code web
sandbox blocks are reachable here.

Run the cells top to bottom. Nothing needs an API key, and nothing trades.

**What to expect:** most strategies will report *no significant edge*. That is the
system working. A backtest that overstates performance is worse than no backtest,
because it converts caution into confidence.


## 1 · Get the code


In [ ]:
!git clone --quiet --branch claude/genjutsu-plugin-install-64u27l https://github.com/Daniel766hi/AI-agent.git engine || echo 'already cloned'
%cd engine
!pip install --quiet -r requirements.txt
print('ready')


### Check the install

If these pass, everything below is running against verified machinery.


In [ ]:
!python run_tests.py


## 2 · Which data sources actually answer?

Binance and Yahoo need no key. CoinGecko works without one but rate limits hard;
set `COINGECKO_API_KEY` in the cell below if you have one.


In [ ]:
import os
# os.environ['COINGECKO_API_KEY'] = 'your-key'   # optional
# os.environ['COINGECKO_PLAN'] = 'demo'          # or 'pro'

!python datacheck.py


## 3 · Fetch real prices

Indonesian equities take a `.JK` suffix. Change these to whatever you want to study.


In [ ]:
import sys; sys.path.insert(0, '.')
from quant import data

IDX = ['BBCA.JK', 'BBRI.JK', 'TLKM.JK', 'ASII.JK', 'BMRI.JK', 'UNVR.JK']
frames, failures = data.fetch_many_yahoo(IDX, range_='5y')

for symbol, frame in frames:
    close = frame['close']
    print(f'{symbol:<10} {len(close):>5} bars  {close.index[0].date()} to {close.index[-1].date()}'
          f'   last {close.iloc[-1]:,.0f}')
for symbol, reason in failures.items():
    print(f'{symbol:<10} FAILED  {reason}')


## 4 · One strategy, one asset

Walk-forward: parameters are fitted on a training window and traded on the window
after, never on the bars that fitted them. The p-value is then charged for every
configuration the search tried.


In [ ]:
from quant.strategies import REGISTRY
from quant.validate import walk_forward, block_bootstrap_pvalue, deflate
from quant.backtest import metrics

symbol, frame = frames[0]
close = frame['close']

fn, grid = REGISTRY['sma_cross']
oos, folds, benchmark, n_trials = walk_forward(close, fn, grid, n_folds=4)
p_raw, edge = block_bootstrap_pvalue(oos, benchmark)

strategy, holding = metrics(oos, periods_per_year=252), metrics(benchmark, periods_per_year=252)
print(f'{symbol} — sma_cross, fitted {folds.iloc[-1]["params"]}\n')
print(f'{"":16}{"strategy":>12}{"buy & hold":>14}')
for key, label in [('cagr','CAGR'), ('sharpe','Sharpe'), ('max_drawdown','Max drawdown')]:
    fmt = '{:>12.2f}' if key == 'sharpe' else '{:>12.1%}'
    print(f'{label:<16}' + fmt.format(strategy[key]) + fmt.replace('12','14').format(holding[key]))
print(f'\nraw p {p_raw:.4f}  ->  {deflate(p_raw, n_trials):.4f} after {n_trials} configs searched')


## 5 · Screen everything at once

Screening N assets with K configs is N×K trials, and at that many attempts
something always looks brilliant. Both p-values are shown: per-asset, and charged
for the whole screen. The gap between them is the point.


In [ ]:
import pandas as pd
from screen import evaluate_all

assets = [(s, f['close']) for s, f in frames]
outcomes = evaluate_all(assets, 'breakout', folds=4)
rows = [o for o in outcomes if o['ok']]

n_combos = rows[0]['n_combos']
total_trials = len(rows) * n_combos
for row in rows:
    row['p_per_asset'] = deflate(row['p_raw'], n_combos)
    row['p_whole_screen'] = deflate(row['p_raw'], total_trials)

table = pd.DataFrame(rows).sort_values('p_raw')
print(f'{len(rows)} assets x {n_combos} configs = {total_trials} trials\n')
display(table[['asset','sharpe','bench_sharpe','total_return','max_dd',
               'p_per_asset','p_whole_screen']].round(4))

survivors = table[table['p_whole_screen'] < 0.05]
print(f'\nlooked significant per-asset : {(table["p_per_asset"] < 0.05).sum()}')
print(f'survived the whole screen   : {len(survivors)}')


### Calibrate it against noise

Run the same screen at the same size on pure random walks. Whatever it finds there,
your setup can produce from nothing — so anything real has to beat it.


In [ ]:
noise = [(f'noise_{i:02d}', data.synthetic(n=len(close), seed=1000+i)['close'])
         for i in range(len(assets))]
noise_rows = [o for o in evaluate_all(noise, 'breakout', folds=4) if o['ok']]
noise_trials = len(noise_rows) * noise_rows[0]['n_combos']

naive = sum(deflate(r['p_raw'], noise_rows[0]['n_combos']) < 0.05 for r in noise_rows)
honest = sum(deflate(r['p_raw'], noise_trials) < 0.05 for r in noise_rows)
print(f'on {len(noise_rows)} pure-noise assets: {naive} looked significant per-asset, '
      f'{honest} survived the screen')


## 6 · Put the best candidate to the desk

Four agents, one mandate each. Approval is unanimous and a veto is final — the
agent that proposes the trade cannot approve it.


In [ ]:
from quant.agents import Desk, Proposal

best = table.iloc[0]['asset']
close_best = dict(assets)[best]

proposal = Proposal(symbol=best, strategy='breakout', close=close_best)
decision = Desk().evaluate(proposal)
print(f'{best}\n')
print(decision.summary())


### If the desk approved what the screen rejected, read this

That can happen, and it is not a contradiction — it is the whole problem in miniature.

The desk sees **one asset**, so it charges the search it can see: the parameter grid,
three configurations. The screen knows you looked at six assets first, so it charges
all eighteen attempts. The same raw result can clear one bar and fail the other.

**The screen's number is the honest one here**, because the only reason this asset
reached the desk is that it won a competition you ran. The desk cannot know that, and
will not protect you from it.

To give the desk a fair question, bring it a strategy and an asset you chose *before*
looking at results — then three trials is the true count and its verdict stands.


## 7 · What it would cost, and what could end it


In [ ]:
from quant.risk import breakeven, cost_drag, kelly_fraction, leverage_table
from quant.backtest import backtest

fn, grid = REGISTRY['breakout']
params = proposal.params or {'window': 55}
result = backtest(close_best, fn(close_best, **params))
stats = breakeven(result, periods_per_year=252)

print(f'{best}, breakout {params}\n')
print(f'round trips per year   {stats["round_trips_per_year"]:>8.1f}')
print(f'annual cost drag       {stats["annual_cost_drag"]:>8.2%}')
print(f'time in market         {stats["time_in_market"]:>8.1%}')
print(f'hurdle vs holding      {stats["hurdle_vs_holding"]:>8.2%}\n')

print('cost drag by trading frequency, at 15bps round trip:')
for label, n in [('monthly',12), ('weekly',52), ('daily',250), ('4x daily',1000)]:
    print(f'  {label:<10}{n:>6} round trips{cost_drag(n):>9.1%}')

print('\nprobability of drawdown within a year:')
display(leverage_table(result['net_return'], periods_per_year=252).round(4))


## What to take from this

**If nothing survived the screen, that is the expected result.** More strategies and
more assets make the multiple-testing problem worse, not better — every extra
attempt is charged against significance, which is why the two p-value columns
diverge as the screen grows.

**Costs are the one guaranteed negative.** At 15 bps a round trip, trading daily
costs about 75% a year against an asset returning 40–60%. Trading less often is the
single highest-value change available, and it requires no edge at all.

**Leverage does not scale outcomes symmetrically.** An account at zero cannot
participate in the recovery, which is why the wipeout column rises so much faster
than the leverage multiple.

---

Nothing here places an order. To paper-trade a strategy, run `trade.py` on a real
machine — Colab is not a place to leave anything running. The live Binance path has
never placed a real order against the real exchange; use testnet first.
